# S01 — Why Deep Learning Works Now

**Module 1**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/dl-f2026-notebooks/blob/main/s01_why_deep_learning_works_now.ipynb)

Every cell below is a worked example from the [S01 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s01/) — same code, same seeds, same outputs. Run them, then change things and see what breaks: that is what this notebook is for.

Slides for this session: [s01.html](https://boyu-zhang-uoi.github.io/dl-f2026/slides/s01.html)


In [ ]:
# Colab only: install PyTorch if it is missing (local runs already have it).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
print("environment ready")

## The perceptron era: a machine that learns


*Expected output starts with:* `linear model  train accuracy: 0.5200`


In [ ]:
import numpy as np

np.random.seed(0)

# XOR-like data: four Gaussian blobs, opposite corners share a label
centers = np.array([[1, 1], [-1, -1], [1, -1], [-1, 1]], dtype=float)
labels = np.array([0, 0, 1, 1])  # (1,1) and (-1,-1) -> class 0; the others -> class 1
X = np.vstack([c + 0.3 * np.random.randn(50, 2) for c in centers])   # (200, 2)
y = np.repeat(labels, 50).astype(float)                              # (200,)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def accuracy(p, y):
    return np.mean((p > 0.5) == y)

# --- Model 1: logistic regression (a linear model) ---
w, b = np.zeros(2), 0.0
lr = 0.5
for step in range(2000):
    p = sigmoid(X @ w + b)                 # predictions, (200,)
    grad_z = (p - y) / len(y)              # dL/dz for cross-entropy loss
    w -= lr * (X.T @ grad_z)
    b -= lr * grad_z.sum()
print(f"linear model  train accuracy: {accuracy(sigmoid(X @ w + b), y):.4f}")

# --- Model 2: one hidden layer of 8 tanh units ---
rng = np.random.default_rng(0)
W1 = 0.5 * rng.standard_normal((2, 8)); b1 = np.zeros(8)
w2 = 0.5 * rng.standard_normal(8);      b2 = 0.0
for step in range(2000):
    H = np.tanh(X @ W1 + b1)               # hidden activations, (200, 8)
    p = sigmoid(H @ w2 + b2)
    grad_z = (p - y) / len(y)              # (200,)
    grad_w2 = H.T @ grad_z
    grad_H = np.outer(grad_z, w2) * (1 - H**2)   # backprop through tanh
    W1 -= lr * (X.T @ grad_H); b1 -= lr * grad_H.sum(axis=0)
    w2 -= lr * grad_w2;        b2 -= lr * grad_z.sum()
H = np.tanh(X @ W1 + b1)
print(f"1-hidden-layer train accuracy: {accuracy(sigmoid(H @ w2 + b2), y):.4f}")

## The feature-engineering detour


*Expected output starts with:* `raw features [x1, x2]        accuracy = 0.5200   weights = [-0.04 -0.07]`


In [ ]:
import numpy as np

np.random.seed(0)

# Same XOR-like data as the first experiment
centers = np.array([[1, 1], [-1, -1], [1, -1], [-1, 1]], dtype=float)
labels = np.array([0, 0, 1, 1])
X = np.vstack([c + 0.3 * np.random.randn(50, 2) for c in centers])
y = np.repeat(labels, 50).astype(float)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def train_logreg(F, y, lr=0.5, steps=2000):
    """Logistic regression by gradient descent on feature matrix F."""
    w, b = np.zeros(F.shape[1]), 0.0
    for _ in range(steps):
        p = sigmoid(F @ w + b)
        g = (p - y) / len(y)
        w -= lr * (F.T @ g)
        b -= lr * g.sum()
    acc = np.mean((sigmoid(F @ w + b) > 0.5) == y)
    return acc, w

feature_sets = {
    "raw features [x1, x2]":        X,
    "engineered [x1, x2, x1*x2]":   np.column_stack([X, X[:, 0] * X[:, 1]]),
}
for name, F in feature_sets.items():
    acc, w = train_logreg(F, y)
    print(f"{name:<28} accuracy = {acc:.4f}   weights = {np.round(w, 2)}")

## The triad: compute, data, algorithms


*Expected output starts with:* `logistic regression  [784, 10]                  7,850 params      15,690 FLOPs/example`


In [ ]:
import numpy as np

def mlp_stats(sizes):
    """Parameter count and forward-pass FLOPs for a fully connected net.

    Each layer computes W @ x + b: (in*out) multiplies + (in*out) adds,
    so ~2*in*out FLOPs per layer per example (bias and activation are
    lower-order terms we fold in as +out each).
    """
    params, flops = 0, 0
    for n_in, n_out in zip(sizes[:-1], sizes[1:]):
        params += n_in * n_out + n_out          # weights + biases
        flops += 2 * n_in * n_out + n_out       # multiply-adds + bias adds
    return params, flops

nets = [
    ("logistic regression", [784, 10]),
    ("small MLP",           [784, 256, 10]),
    ("wider + deeper MLP",  [784, 1024, 1024, 10]),
]
for name, sizes in nets:
    p, f = mlp_stats(sizes)
    print(f"{name:<20} {str(sizes):<22} {p:>9,d} params  {f:>10,d} FLOPs/example")

# Training cost = forward + backward (~2x forward) per example, per epoch.
p, f = mlp_stats([784, 1024, 1024, 10])
train_flops = 3 * f * 60_000 * 10               # 3x forward cost, 60k examples, 10 epochs
print(f"\ntraining the deeper net, 10 epochs on 60k examples: ~{train_flops:.2e} FLOPs")
print(f"at 10^12 FLOP/s (one modest core doing ~1 TFLOP/s): ~{train_flops / 1e12:.1f} s of compute")

## Data: measuring what more examples buy


*Expected output starts with:* `n_train =    50: train acc 1.0000   test acc 0.9200`


In [ ]:
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

def make_spirals(n_per_class, rng, noise=0.25):
    """Two interleaved spirals -- a nonlinear problem with a fixed Bayes rate."""
    X, y = [], []
    for c in range(2):
        t = rng.uniform(0.25, 3.0, size=n_per_class)
        ang = 2.5 * t + np.pi * c
        pts = np.stack([t * np.cos(ang), t * np.sin(ang)], axis=1)
        pts += noise * rng.standard_normal(pts.shape)
        X.append(pts)
        y.append(np.full(n_per_class, c))
    return np.vstack(X).astype(np.float32), np.concatenate(y)

def train_mlp(X, y, steps=2000, lr=0.01):
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(2, 64), nn.ReLU(),
                          nn.Linear(64, 64), nn.ReLU(),
                          nn.Linear(64, 2))
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    Xt, yt = torch.from_numpy(X), torch.from_numpy(y)
    for _ in range(steps):
        opt.zero_grad()
        loss = nn.functional.cross_entropy(model(Xt), yt)
        loss.backward()
        opt.step()
    return model

# One fixed, large test set; training sets of growing size, same distribution
X_test, y_test = make_spirals(2000, np.random.default_rng(12345))

for n in [25, 100, 400, 1600]:
    X_tr, y_tr = make_spirals(n, np.random.default_rng(0))
    model = train_mlp(X_tr, y_tr)
    with torch.no_grad():
        acc_tr = (model(torch.from_numpy(X_tr)).argmax(1).numpy() == y_tr).mean()
        acc_te = (model(torch.from_numpy(X_test)).argmax(1).numpy() == y_test).mean()
    print(f"n_train = {2 * n:>5}: train acc {acc_tr:.4f}   test acc {acc_te:.4f}")

## Try it yourself

1. In the XOR experiment, reduce the hidden layer from 8 units to 2, then to 1. At what width does the network stop reaching 100% train accuracy? Two hidden units are enough in principle — do they work with this initialization every time? Try a few seeds.
2. Move the four cluster centers so that opposite corners no longer share a label (e.g., left pair class 0, right pair class 1). Rerun both models and explain the linear model's new accuracy.
3. In the feature-engineering experiment, replace the `x1*x2` feature with `x1^2` and with `|x1 - x2|`. Which engineered features solve XOR and which do not? State the general property a single added feature needs.
4. Extend `mlp_stats` to report bytes of memory for parameters in float32, and estimate for the `[784, 1024, 1024, 10]` net how many examples per second one training step could process at 10^12 FLOP/s.
5. Using the rule of thumb `3 * forward FLOPs` per training example, estimate the total FLOPs to train a 60-million-parameter network for one epoch on 1.2 million images, assuming forward cost ~`2 * params` FLOPs per image (a loose lower bound that ignores weight sharing). Compare with the seconds-scale number from our MLP.
6. In the spiral experiment, hold the training set fixed at 200 examples and instead scale the *model*: hidden widths 4, 16, 64, 256. Does more capacity help, hurt, or neither at this data size? Reconcile what you see with the data-scaling table.


---

Full discussion of everything above: [S01 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s01/).
